In [1]:
import findspark
findspark.init()

from pyspark.conf import SparkConf
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

conf = SparkConf().setAppName("602").setMaster("local[4]")
spark = SparkSession.builder.config(conf = conf).getOrCreate()
spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/08/25 01:22:31 WARN Utils: Your hostname, de24, resolves to a loopback address: 127.0.1.1; using 192.168.29.229 instead (on interface enp0s3)
25/08/25 01:22:31 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/08/25 01:22:34 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [ ]:
'''
Table: RequestAccepted

+----------------+---------+
| Column Name    | Type    |
+----------------+---------+
| requester_id   | int     |
| accepter_id    | int     |
| accept_date    | date    |
+----------------+---------+
(requester_id, accepter_id) is the primary key 
(combination of columns with unique values) for this table.
This table contains the ID of the user who sent the request, 
the ID of the user who received the request, and the date when the request was accepted.
 

Write a solution to find the people who have the most friends and the most friends number.

The test cases are generated so that only one person has the most friends.

The result format is in the following example.

 

Example 1:

Input: 
RequestAccepted table:
+--------------+-------------+-------------+
| requester_id | accepter_id | accept_date |
+--------------+-------------+-------------+
| 1            | 2           | 2016/06/03  |
| 1            | 3           | 2016/06/08  |
| 2            | 3           | 2016/06/08  |
| 3            | 4           | 2016/06/09  |
+--------------+-------------+-------------+
Output: 
+----+-----+
| id | num |
+----+-----+
| 3  | 3   |
+----+-----+
Explanation: 
The person with id 3 is a friend of people 1, 2, and 4, 
so he has three friends in total, which is the most number than any others.
'''

In [3]:
data = [
(1,2,'2016/06/03'),
(1,3,'2016/06/08'),
(2,3,'2016/06/08'),
(3,4,'2016/06/09'),
]
schema = ['requester_id','accepter_id','accept_date']

In [4]:
df = spark.createDataFrame(data = data, schema = schema)
df.show()

+------------+-----------+-----------+
|requester_id|accepter_id|accept_date|
+------------+-----------+-----------+
|           1|          2| 2016/06/03|
|           1|          3| 2016/06/08|
|           2|          3| 2016/06/08|
|           3|          4| 2016/06/09|
+------------+-----------+-----------+



In [7]:
df_union = (df.groupBy(F.col("requester_id"))\
             .agg(F.col("requester_id").alias("id"),
                  F.count("*").alias("num"))).union(
            df.groupBy(F.col("accepter_id"))\
             .agg(F.col("accepter_id").alias("id"),
                  F.count("*").alias("num"))
                  )

df_union.groupBy(F.col("id"))\
        .agg(F.sum(F.col("num")).alias("num"))\
        .orderBy(F.col("num").desc())\
        .show(1)

[Stage 21:===========================================>              (3 + 1) / 4]

+---+---+
| id|num|
+---+---+
|  3|  3|
+---+---+
only showing top 1 row


## SQL Solution
<pre>
WITH T3_UNION as (
SELECT requester_id as id, count(*) as num 
FROM RequestAccepted 
GROUP BY requester_id
UNION
SELECT accepter_id as id, count(*) as num 
FROM RequestAccepted 
GROUP BY accepter_id
)
SELECT id, SUM(num) as num
FROM T3_UNION
GROUP BY id
ORDER BY  num limit 1
</pre>